# Hospital Patient Data Exploration

## Overview
This notebook performs exploratory data analysis (EDA) on hospital patient data to:
- Understand data structure and content
- Identify missing values and data quality issues
- Detect outliers and anomalies
- Document findings for data cleaning

**Period**: July 2024 - July 2025  
**Dataset**: 5,000+ patient records with admissions and treatments

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

print("✓ All libraries loaded successfully!")

## 2. Connect to Database

In [ ]:
# Setup database path
DB_PATH = os.path.join('..', 'data', 'hospital.db')

# Connect to database
if os.path.exists(DB_PATH):
    conn = sqlite3.connect(DB_PATH)
    print(f"✓ Connected to database: {DB_PATH}")
else:
    print(f"✗ Database not found at {DB_PATH}")
    print("Please run: python ../scripts/generate_hospital_data.py")
    print("Then run: python ../scripts/load_data.py")

## 3. Load Data

In [ ]:
# Load raw data from database
patients = pd.read_sql_query("SELECT * FROM patients", conn)
admissions = pd.read_sql_query("SELECT * FROM admissions", conn)
treatments = pd.read_sql_query("SELECT * FROM treatments", conn)
departments = pd.read_sql_query("SELECT * FROM departments", conn)

print("Data loaded successfully!")
print(f"\nPatients: {len(patients):,} records")
print(f"Admissions: {len(admissions):,} records")
print(f"Treatments: {len(treatments):,} records")
print(f"Departments: {len(departments)} departments")

## 4. Patients Data Quality

In [ ]:
print("=" * 60)
print("PATIENTS DATA QUALITY ASSESSMENT")
print("=" * 60)

# Display basic info
print("\nData Types:")
print(patients.dtypes)

print("\nFirst few records:")
print(patients.head())

print("\nBasic Statistics:")
print(patients.describe())

In [ ]:
# Missing values
print("\nMissing Values:")
missing = patients.isnull().sum()
missing_pct = (patients.isnull().sum() / len(patients) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print("\nGender Distribution:")
print(patients['gender'].value_counts())

print("\nAge Distribution:")
print(patients['date_of_birth'].describe())

## 5. Admissions Data Quality

In [ ]:
print("=" * 60)
print("ADMISSIONS DATA QUALITY ASSESSMENT")
print("=" * 60)

print("\nFirst few records:")
print(admissions.head())

# Convert date columns
admissions['admission_date'] = pd.to_datetime(admissions['admission_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])

# Check for invalid dates
invalid_dates = admissions[admissions['discharge_date'] < admissions['admission_date']]
print(f"\nInvalid date sequences (discharge < admission): {len(invalid_dates)}")

# Calculate length of stay
admissions['los'] = (admissions['discharge_date'] - admissions['admission_date']).dt.days

print("\nLength of Stay Statistics (days):")
print(admissions['los'].describe())

# Negative LOS
negative_los = admissions[admissions['los'] < 0]
print(f"\nNegative LOS records: {len(negative_los)}")

In [ ]:
# Department analysis
print("\nAdmissions by Department:")
dept_admissions = admissions['department_id'].value_counts().sort_index()
print(dept_admissions)

# Status distribution
print("\nAdmission Status:")
print(admissions['status'].value_counts())

# Reason distribution
print("\nTop 10 Admission Reasons:")
print(admissions['admission_reason'].value_counts().head(10))

## 6. Treatments Data Quality

In [ ]:
print("=" * 60)
print("TREATMENTS DATA QUALITY ASSESSMENT")
print("=" * 60)

print("\nFirst few records:")
print(treatments.head())

# Treatment type distribution
print("\nTreatment Types:")
print(treatments['treatment_type'].value_counts())

# Cost analysis
print("\nTreatment Cost Statistics ($):")
print(treatments['treatment_cost'].describe())

In [ ]:
# Effectiveness scores
print("\nEffectiveness Score Statistics:")
print(treatments['effectiveness_score'].describe())

# Missing effectiveness scores
missing_effectiveness = treatments['effectiveness_score'].isnull().sum()
print(f"\nMissing effectiveness scores: {missing_effectiveness}")

# Outcome distribution
print("\nTreatment Outcomes:")
print(treatments['outcome'].value_counts())

# Recovery rate
recovered = treatments[treatments['outcome'].isin(['Recovered', 'Improved'])]
recovery_rate = len(recovered) / len(treatments) * 100
print(f"\nRecovery Rate (Recovered + Improved): {recovery_rate:.2f}%")

## 7. Outlier Detection

In [ ]:
# Cost outliers
Q1 = treatments['treatment_cost'].quantile(0.25)
Q3 = treatments['treatment_cost'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = treatments[(treatments['treatment_cost'] < lower_bound) | (treatments['treatment_cost'] > upper_bound)]
print(f"Cost outliers (IQR method): {len(outliers)} ({len(outliers)/len(treatments)*100:.2f}%)")
print(f"  - Lower bound: ${lower_bound:,.2f}")
print(f"  - Upper bound: ${upper_bound:,.2f}")

# LOS outliers
Q1_los = admissions['los'].quantile(0.25)
Q3_los = admissions['los'].quantile(0.75)
IQR_los = Q3_los - Q1_los
upper_los = Q3_los + 1.5 * IQR_los

los_outliers = admissions[admissions['los'] > upper_los]
print(f"\nLOS outliers (> {upper_los:.0f} days): {len(los_outliers)} ({len(los_outliers)/len(admissions)*100:.2f}%)")

## 8. Data Relationships

In [ ]:
# Treatments per admission
treatments_per_admission = treatments.groupby('admission_id').size()
print("Treatments per Admission:")
print(treatments_per_admission.describe())

# Admissions per patient
admissions_per_patient = admissions.groupby('patient_id').size()
print("\nAdmissions per Patient:")
print(admissions_per_patient.describe())

# Readmission analysis
readmitted = admissions_per_patient[admissions_per_patient > 1]
readmission_rate = len(readmitted) / len(patients) * 100
print(f"\nReadmission rate: {readmission_rate:.2f}%")
print(f"Patients with multiple admissions: {len(readmitted):,} ({len(readmitted)/len(patients)*100:.2f}%)")

## 9. Visualization: Data Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender distribution
patients['gender'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['#1f77b4', '#ff7f0e'])
axes[0, 0].set_title('Gender Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Count')

# Length of stay distribution
axes[0, 1].hist(admissions['los'], bins=50, color='#2ca02c', edgecolor='black')
axes[0, 1].set_title('Length of Stay Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Days')
axes[0, 1].set_ylabel('Frequency')

# Treatment cost distribution
axes[1, 0].hist(treatments['treatment_cost'], bins=50, color='#d62728', edgecolor='black')
axes[1, 0].set_title('Treatment Cost Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Cost ($)')
axes[1, 0].set_ylabel('Frequency')

# Outcome distribution
treatments['outcome'].value_counts().plot(kind='barh', ax=axes[1, 1], color='#9467bd')
axes[1, 1].set_title('Treatment Outcomes', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('../output/01_data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to ../output/01_data_distribution.png")

## 10. Summary & Recommendations

In [ ]:
print("\n" + "="*60)
print("DATA EXPLORATION SUMMARY")
print("="*60)

print("\n✓ KEY FINDINGS:")
print(f"  1. Patient Records: {len(patients):,} patients")
print(f"  2. Admission Records: {len(admissions):,} admissions")
print(f"  3. Treatment Records: {len(treatments):,} treatments")
print(f"  4. Average LOS: {admissions['los'].mean():.2f} days")
print(f"  5. Overall Recovery Rate: {recovery_rate:.2f}%")
print(f"  6. Readmission Rate: {readmission_rate:.2f}%")
print(f"  7. Cost Outliers: {len(outliers)} treatment records")
print(f"  8. Invalid Sequences: {len(invalid_dates)} date inconsistencies")

print("\n⚠ DATA QUALITY ISSUES TO ADDRESS:")
print(f"  1. {len(invalid_dates)} admissions with invalid date sequences")
print(f"  2. {len(negative_los)} admissions with negative LOS")
print(f"  3. {len(outliers)} extreme cost outliers")
print(f"  4. {missing_effectiveness} missing effectiveness scores")

print("\n→ NEXT STEPS:")
print("  1. Run data cleaning script: python ../scripts/data_cleaning.py")
print("  2. Review cleaned data in 02_analysis_insights.ipynb")
print("  3. Generate visualizations in 03_visualizations.ipynb")

conn.close()
print("\n✓ Exploration completed!")